<a href="https://colab.research.google.com/github/bhaviii123/Air_Passenger_Forecasting_ML_vs_DL.ipynb/blob/main/different_epochs_and_batch_sizes_using_the_NYC_TLC_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To experiment with different epochs and batch sizes using the NYC TLC dataset, we will use Pandas for data handling and TensorFlow/Keras for the model.

Step 1: Load and Clean the Data

In [1]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load a sample (Yellow Taxi Data)
df = pd.read_parquet('yellow_tripdata_2024-01.parquet')

# Feature Engineering: Convert timestamps to usable numbers
df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
df['day_of_week'] = df['tpep_pickup_datetime'].dt.dayofweek

# Filter for realistic trips (Distance > 0 and Fare between $2 and $100)
df = df[(df['trip_distance'] > 0) & (df['total_amount'].between(2, 100))].sample(100000)

# Select Features and Target (Predicting total_amount)
features = ['trip_distance', 'pickup_hour', 'day_of_week', 'PULocationID', 'DOLocationID']
X = df[features]
y = df['total_amount']

# Split and Scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

--2026-03-09 14:22:52--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 52.85.97.61, 52.85.97.13, 52.85.97.44, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.85.97.61|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 49961641 (48M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2024-01.parquet’

yellow_tripdata_202 100%[===================>]  47.65M   126MB/s    in 0.4s    

2026-03-09 14:22:52 (126 MB/s) - ‘yellow_tripdata_2024-01.parquet’ saved [49961641/49961641]



Step 2: Define the Experiment Function

In [2]:
import tensorflow as tf
from tensorflow.keras import layers

def run_experiment(batch_size, epochs):
    print(f"\n--- Running: Batch Size {batch_size}, Epochs {epochs} ---")

    model = tf.keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
        layers.Dense(32, activation='relu'),
        layers.Dense(1) # Output layer for regression
    ])

    model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    history = model.fit(
        X_train, y_train,
        validation_split=0.2,
        batch_size=batch_size,
        epochs=epochs,
        verbose=0 # Set to 1 to see progress bars
    )

    val_mae = history.history['val_mae'][-1]
    print(f"Final Validation MAE: ${val_mae:.2f}")
    return history

Step 3: Run Multiple Trials

In [ ]:
# Define our test grid
batch_options = [32, 256, 1024]
epoch_options = [10, 30]

results = {}

for b in batch_options:
    for e in epoch_options:
        history = run_experiment(b, e)
        results[f"B{b}_E{e}"] = history.history['val_loss'][-1]

# Find the best combination
best_config = min(results, key=results.get)
print(f"\nBest Configuration: {best_config} with Loss: {results[best_config]}")


--- Running: Batch Size 32, Epochs 10 ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Final Validation MAE: $3.35

--- Running: Batch Size 32, Epochs 30 ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Step 4: The "Smart" Training Function

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

def run_smart_experiment(batch_size):
    print(f"\n--- Testing Batch Size: {batch_size} (Max Epochs: 100) ---")

    # Define Early Stopping
    # monitor='val_loss': Watch the validation error
    # patience=5: If it doesn't improve for 5 rounds, stop.
    # restore_best_weights=True: Go back to the version that had the lowest error.
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    model = tf.keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
        layers.Dense(64, activation='relu'),
        layers.Dense(1)
    ])

    model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    history = model.fit(
        X_train, y_train,
        validation_split=0.2,
        batch_size=batch_size,
        epochs=100, # Set a high max, let EarlyStopping handle the rest
        callbacks=[early_stop],
        verbose=1
    )

    return history

# Run for two different scales
history_small = run_smart_experiment(32)
history_large = run_smart_experiment(2048)

Step 5: Compare and Visualize

In [ ]:
def plot_history(history, title):
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title(title)
    plt.xlabel('Epochs')
    plt.ylabel('Mean Squared Error')
    plt.legend()
    plt.show()

plot_history(history_small, "Small Batch (32) Results")
plot_history(history_large, "Large Batch (2048) Results")